# TA-RecMind V2 — Intra-Layer Gated Fusion

Kiến trúc cập nhật theo phân tích toàn diện: intra-layer gated fusion, gate đối xứng user-item, negative sampling 3-thành-phần, stratified evaluation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0: CÀI ĐẶT & BIẾN MÔI TRƯỜNG
# ═══════════════════════════════════════════════════════════════════════════════
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q sentence-transformers huggingface_hub datasets

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: IMPORTS & CẤU HÌNH TRUNG TÂM
# ═══════════════════════════════════════════════════════════════════════════════
import gc, json, time, threading, subprocess, psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from huggingface_hub import hf_hub_download
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Mount Google Drive ──────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
    print("[INFO] Google Drive mounted.")
except ImportError:
    IN_COLAB = False
    print("[WARN] Không có môi trường Colab. Chạy local.")

_DRIVE_ROOT = "/content/drive/MyDrive/tarecmind"

CFG = {
    "REPO_ID":   "chuongdo1104/amazon-2023-gold",
    "DRIVE_ROOT": _DRIVE_ROOT,
    "DATA_DIR":   f"{_DRIVE_ROOT}/data",
    "SAVE_DIR":   f"{_DRIVE_ROOT}/weights",
    "CACHE_DIR":  "/content/recsys_cache",

    # Model
    "EMBED_DIM":   128,
    "LLM_DIM":     384,
    "GCN_LAYERS":  2,
    "TEMPERATURE": 0.15,

    # Training
    "EPOCHS":          150,
    "LR_JOINT":        1e-3,
    "WEIGHT_DECAY":    1e-4,
    "GRAD_CLIP":       1.0,

    # Loss weights — V2: tăng lambda_align, giảm noise_scale
    "LAMBDA_ALIGN":   0.3,
    "LAMBDA_CL":      0.05,
    "NOISE_SCALE":    0.08,
    "LAMBDA_TEMPORAL": 0.5,   # decay Electronics: nhẹ hơn V1

    # Evaluation
    "PATIENCE":        15,
    "EVAL_EVERY":      2,      # eval mỗi 2 epoch để tiết kiệm thời gian
    "EVAL_STRAT_N":    1000,   # số users stratified evaluation

    # Encoding LLM
    "ENCODE_CHUNK":  30000,
    "ENCODE_BATCH":  256,

    "SAVE_EVERY_EPOCH": True,
    "KEEPALIVE_MINS":   25,
}

for d in [CFG["DATA_DIR"], CFG["SAVE_DIR"], CFG["CACHE_DIR"]]:
    os.makedirs(d, exist_ok=True)

print("[INFO] Cấu hình V2 đã sẵn sàng.")

Mounted at /content/drive
[INFO] Google Drive mounted.
[INFO] Cấu hình V2 đã sẵn sàng.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: PHÁT HIỆN GPU & ĐIỀU CHỈNH BATCH SIZE
# ═══════════════════════════════════════════════════════════════════════════════
def detect_and_adjust_gpu():
    sys_ram_gb = psutil.virtual_memory().total / 1e9
    print(f"[INFO] System RAM: {sys_ram_gb:.1f} GB")

    if not torch.cuda.is_available():
        print("[WARN] Không có GPU.")
        return "cpu"

    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")

    if "A100" in gpu_name or vram_gb > 35:
        CFG["BATCH_SIZE"]     = 16384
        CFG["ALIGN_SUBBATCH"] = 4096
        CFG["ACCUM_STEPS"]    = 1
        CFG["ENCODE_BATCH"]   = 1024
        print("   → Chế độ A100: batch=16384, subbatch=4096")
    elif vram_gb > 16:
        CFG["BATCH_SIZE"]     = 8192
        CFG["ALIGN_SUBBATCH"] = 2048
        CFG["ACCUM_STEPS"]    = 1
        CFG["ENCODE_BATCH"]   = 512
        print("   → Chế độ L4/V100: batch=8192, subbatch=2048")
    else:
        CFG["BATCH_SIZE"]     = 2048
        CFG["ALIGN_SUBBATCH"] = 512
        CFG["ACCUM_STEPS"]    = 4
        CFG["ENCODE_BATCH"]   = 256
        print("   → Chế độ T4: batch=2048, accum=4")

    return "cuda"

device = detect_and_adjust_gpu()

# ── Keep-alive thread ────────────────────────────────────────────────────────
_keepalive_stop = threading.Event()
def _keepalive_worker():
    interval = CFG["KEEPALIVE_MINS"] * 60
    ka_file  = os.path.join(CFG["CACHE_DIR"], "keepalive.txt")
    while not _keepalive_stop.is_set():
        time.sleep(interval)
        if not _keepalive_stop.is_set():
            with open(ka_file, "w") as f:
                f.write(f"alive {time.strftime('%H:%M:%S')}\n")

threading.Thread(target=_keepalive_worker, daemon=True).start()
print(f"[INFO] Keep-alive thread started. Device: {device}")

[INFO] System RAM: 89.6 GB
[INFO] GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB
   → Chế độ A100: batch=16384, subbatch=4096
[INFO] Keep-alive thread started. Device: cuda


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: TẢI DỮ LIỆU & THÔNG TIN GRAPH
# Kết quả được lưu vào Drive → tái sử dụng không cần tải lại
# ═══════════════════════════════════════════════════════════════════════════════
def download_hf(filename, local_dir=None):
    target     = local_dir or CFG["CACHE_DIR"]
    local_path = os.path.join(target, os.path.basename(filename))
    if os.path.exists(local_path):
        return local_path
    print(f"[INFO] Đang tải {filename}...")
    return hf_hub_download(repo_id=CFG["REPO_ID"], filename=filename,
                           repo_type="dataset", local_dir=target)

print("\n--- TẢI METADATA ---")
with open(download_hf("gold/gold_dataset_stats.json"), "r") as f:
    dataset_stats = json.load(f)

num_users   = dataset_stats["n_users"]
num_items   = dataset_stats["n_items"]
total_nodes = num_users + num_items
print(f"[INFO] {num_users:,} users | {num_items:,} items | {total_nodes:,} nodes tổng")

# ── Load edge index & frequency arrays ──────────────────────────────────────
edge_index_raw  = np.load(download_hf("gold/gold_edge_index.npy"))
item_train_freq = np.load(download_hf("gold/gold_item_train_freq.npy"))
user_train_freq = np.load(download_hf("gold/gold_user_train_freq.npy"))    # V2 MỚI

item_train_freq_t = torch.from_numpy(item_train_freq).float()
user_train_freq_t = torch.from_numpy(user_train_freq).float()               # V2 MỚI
train_edge_index  = torch.from_numpy(edge_index_raw).long()
n_train_edges     = train_edge_index.shape[1]

# ── Tail edge indices (oversampling) ─────────────────────────────────────────
tail_item_mask    = (item_train_freq_t <= 5) & (item_train_freq_t > 0)
is_tail_edge      = tail_item_mask[train_edge_index[1]]
tail_edge_indices = torch.where(is_tail_edge)[0]

# ── Popularity sampling weights (cho negative sampling thành phần 2) ─────────
neg_prob_raw   = np.load(download_hf("gold/gold_negative_sampling_prob.npy"))
neg_prob_raw   = neg_prob_raw / neg_prob_raw.sum()
neg_prob_t     = torch.from_numpy(neg_prob_raw).float()

assert n_train_edges > 0
assert len(item_train_freq) == num_items
assert len(user_train_freq) == num_users

print(f"[INFO] {n_train_edges:,} edges | {len(tail_edge_indices):,} tail edges")
print(f"[SUCCESS] Dữ liệu graph đã sẵn sàng.")

del item_train_freq, user_train_freq, edge_index_raw, neg_prob_raw
gc.collect()


--- TẢI METADATA ---
[INFO] Đang tải gold/gold_dataset_stats.json...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


gold_dataset_stats.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

[INFO] 1,483,920 users | 1,610,012 items | 3,093,932 nodes tổng
[INFO] Đang tải gold/gold_edge_index.npy...


gold/gold_edge_index.npy:   0%|          | 0.00/178M [00:00<?, ?B/s]

...

gold/gold_user_train_freq.npy:   0%|          | 0.00/11.9M [00:00<?, ?B/s]

[INFO] Đang tải gold/gold_negative_sampling_prob.npy...


gold/gold_negative_sampling_prob.npy:   0%|          | 0.00/6.44M [00:00<?, ?B/s]

[INFO] 11,123,425 edges | 1,262,343 tail edges
[SUCCESS] Dữ liệu graph đã sẵn sàng.


167

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: LLM EMBEDDINGS — DRIVE-BACKED CACHE (chunk-by-chunk, chống OOM)
# Lưu từng chunk lên Drive → an toàn khi Colab bị ngắt
# ═══════════════════════════════════════════════════════════════════════════════
FORCE_ENCODE = False   # Đặt True nếu cần encode lại từ đầu

def drive_path_emb(prefix):
    return os.path.join(CFG["DATA_DIR"], f"gold_{prefix}_embeddings.npy")

def local_path_emb(prefix):
    return os.path.join(CFG["CACHE_DIR"], f"gold_{prefix}_embeddings.npy")

need_item_emb = FORCE_ENCODE or not os.path.exists(drive_path_emb("item"))
need_user_emb = FORCE_ENCODE or not os.path.exists(drive_path_emb("user"))

if need_item_emb or need_user_emb:
    gc.collect(); torch.cuda.empty_cache()
    print(f"[INFO] Khởi tạo SentenceTransformer...")
    model_st = SentenceTransformer("all-MiniLM-L6-v2", device=device)
    if device == "cuda":
        model_st = model_st.half()

    def encode_and_save(texts, prefix):
        total_len = len(texts)
        parts = []
        for i in range(0, total_len, CFG["ENCODE_CHUNK"]):
            end  = min(i + CFG["ENCODE_CHUNK"], total_len)
            ckpt = os.path.join(CFG["DATA_DIR"], f"ckpt_{prefix}_{i}_{end}.npy")
            if os.path.exists(ckpt):
                print(f"[INFO] Skip {prefix} {i:,}-{end:,} (cache tồn tại).")
                parts.append(np.load(ckpt))
            else:
                print(f"[INFO] Encoding {prefix} {i:,}-{end:,}...")
                chunk_emb = model_st.encode(
                    texts[i:end],
                    batch_size=CFG["ENCODE_BATCH"],
                    show_progress_bar=True,
                    normalize_embeddings=True,
                    convert_to_numpy=True
                )
                np.save(ckpt, chunk_emb)
                parts.append(chunk_emb)
                gc.collect()
        final = np.vstack(parts)
        np.save(drive_path_emb(prefix), final)
        print(f"[SUCCESS] Đã encode & lưu {prefix} lên Drive.")

    if need_item_emb:
        df_item_text = load_dataset("chuongdo1104/amazon-2023-silver",
                                    data_dir="silver/silver_item_text_profile.parquet",
                                    split="train").to_pandas()
        df_item_map  = pd.read_parquet(download_hf("gold/gold_item_id_map.parquet",
                                                    local_dir=CFG["CACHE_DIR"]))

        df_item_map["parent_asin"] = df_item_map["parent_asin"].astype(str).str.strip()
        df_item_text["parent_asin"] = df_item_text["parent_asin"].astype(str).str.strip()

        item_merged  = df_item_map.merge(df_item_text, on="parent_asin", how="left")


        empty_cnt = item_merged["item_text"].isna().sum()
        if empty_cnt > 0.1 * len(item_merged):
            print(f"[CẢNH BÁO ĐỎ] ID Mapping thất bại! Có {empty_cnt:,} items không match được Text.")
        else:
            print(f"[OK] Đã merge text thành công.")





        item_texts   = item_merged.sort_values("item_idx")["item_text"].fillna("").tolist()


        del df_item_text, df_item_map, item_merged; gc.collect()
        encode_and_save(item_texts, "item")
        del item_texts; gc.collect()

    if need_user_emb:
        df_user_text = load_dataset("chuongdo1104/amazon-2023-silver",
                                    data_dir="silver/silver_user_text_profile.parquet",
                                    split="train").to_pandas()
        df_user_map  = pd.read_parquet(download_hf("gold/gold_user_id_map.parquet",
                                                    local_dir=CFG["CACHE_DIR"]))
        user_merged  = df_user_map.merge(df_user_text, on="reviewer_id", how="left")
        user_texts   = user_merged.sort_values("user_idx")["user_text"].fillna("").tolist()
        del df_user_text, df_user_map, user_merged; gc.collect()
        encode_and_save(user_texts, "user")
        del user_texts; gc.collect()

    del model_st; gc.collect(); torch.cuda.empty_cache()
else:
    print("[INFO] LLM Embeddings đã có trên Drive.")

# ── Copy từ Drive về local SSD để tăng tốc I/O ──────────────────────────────
print("[INFO] Copy embeddings Drive → SSD local...")
for prefix in ["user", "item"]:
    local_p = local_path_emb(prefix)
    drive_p = drive_path_emb(prefix)
    if not os.path.exists(local_p):
        subprocess.run(["cp", drive_p, local_p], check=True)

user_emb_llm = torch.from_numpy(np.load(local_path_emb("user"))).float()
item_emb_llm = torch.from_numpy(np.load(local_path_emb("item"))).float()
print(f"[READY] LLM Embeddings: User {user_emb_llm.shape} | Item {item_emb_llm.shape}")

[INFO] LLM Embeddings đã có trên Drive.
[INFO] Copy embeddings Drive → SSD local...
[READY] LLM Embeddings: User torch.Size([1483920, 384]) | Item torch.Size([1610012, 384])


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: XỬ LÝ GRAPH — TRAIN/VAL SPLIT & SPARSE ADJACENCY MATRIX
# Lưu kết quả vào Drive để tránh tính lại khi resume
# ═══════════════════════════════════════════════════════════════════════════════
path_train_edges = os.path.join(CFG["DATA_DIR"], "train_edges.pt")
path_val_edges   = os.path.join(CFG["DATA_DIR"], "val_edges.pt")
path_sparse_adj  = os.path.join(CFG["DATA_DIR"], "sparse_adj.pt")

if all(os.path.exists(p) for p in [path_train_edges, path_val_edges, path_sparse_adj]):
    print("[INFO] Tải graph structures từ Drive...")
    train_edge_index = torch.load(path_train_edges, map_location=device, weights_only=True)
    val_edges_t      = torch.load(path_val_edges,   map_location="cpu",  weights_only=True)
    sparse_adj       = torch.load(path_sparse_adj,  map_location=device, weights_only=True)
    n_train_edges    = train_edge_index.shape[1]
    VAL_SIZE         = val_edges_t.shape[1]
else:
    print("[INFO] Tạo mới graph structures...")
    num_edges_total = n_train_edges
    VAL_SIZE        = min(5000, num_edges_total // 20)
    val_idx         = np.random.choice(num_edges_total, VAL_SIZE, replace=False)
    val_edges_t     = train_edge_index[:, val_idx].cpu().long()

    train_mask           = np.ones(num_edges_total, dtype=bool)
    train_mask[val_idx]  = False
    train_edge_index     = train_edge_index[:, train_mask].to(device)
    n_train_edges        = train_edge_index.shape[1]

    # Chuẩn hóa đối xứng Â = D^{-0.5} A D^{-0.5}
    row = torch.cat([train_edge_index[0], train_edge_index[1] + num_users])
    col = torch.cat([train_edge_index[1] + num_users, train_edge_index[0]])
    gei = torch.stack([row, col]).long()
    deg = torch.bincount(gei[0], minlength=total_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
    # float16 để tiết kiệm VRAM
    ew  = (deg_inv_sqrt[gei[0]] * deg_inv_sqrt[gei[1]]).half()
    adj = torch.sparse_coo_tensor(gei, ew, (total_nodes, total_nodes))
    sparse_adj = adj.to(device).to_sparse_csr()

    del adj, ew, deg, deg_inv_sqrt, gei, row, col, train_mask
    gc.collect(); torch.cuda.empty_cache()

    torch.save(train_edge_index, path_train_edges)
    torch.save(val_edges_t,      path_val_edges)
    torch.save(sparse_adj,       path_sparse_adj)
    print("[INFO] Đã lưu graph structures lên Drive.")

# ── Đưa frequency tensors lên GPU ───────────────────────────────────────────
item_train_freq_t = item_train_freq_t.to(device)
user_train_freq_t = user_train_freq_t.to(device)
neg_prob_t        = neg_prob_t.to(device)

# ── Cập nhật tail_edge_indices theo train_edge_index mới ────────────────────
edge_item_freqs   = item_train_freq_t[train_edge_index[1].long()]
tail_edge_indices = torch.nonzero(edge_item_freqs <= 5).squeeze()

# ── Chuẩn hóa L2 LLM embeddings và đẩy lên GPU ──────────────────────────────
print("[INFO] Normalize & copy LLM embeddings lên GPU...")
user_emb_llm = F.normalize(user_emb_llm.to(device).half(), p=2, dim=1)
item_emb_llm = F.normalize(item_emb_llm.to(device).half(), p=2, dim=1)

assert user_emb_llm.shape[0] == num_users
assert item_emb_llm.shape[0] == num_items

gc.collect()
print(f"[SUCCESS] Graph ready. Train edges: {n_train_edges:,} | Val: {VAL_SIZE:,}")
print(f"[INFO] Tail edges: {len(tail_edge_indices):,}")

[INFO] Tải graph structures từ Drive...
[INFO] Normalize & copy LLM embeddings lên GPU...
[SUCCESS] Graph ready. Train edges: 11,118,425 | Val: 5,000
[INFO] Tail edges: 1,261,762


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: KIẾN TRÚC TARecMind V2 — INTRA-LAYER GATED FUSION
#
# Thay đổi so với V1:
#   • forward_gcn_gated(): gate được tính TRƯỚC mỗi lần nhân ma trận (intra-layer)
#   • _compute_gate(): đối xứng user-item, dùng MLP phi tuyến thay vì cộng tuyến tính
#   • w_niche: tham số học được giảm gamma cho user thích hàng ngách
#   • Cold-start hard masking: freq=0 → gamma=0 → 100% LLM
# ═══════════════════════════════════════════════════════════════════════════════
class TARecMindV2(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=128, llm_dim=384, gcn_layers=2):
        super().__init__()
        self.num_users  = num_users
        self.num_items  = num_items
        self.gcn_layers = gcn_layers

        self.user_id_emb = nn.Embedding(num_users, embed_dim)
        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        nn.init.normal_(self.user_id_emb.weight, std=0.1)
        nn.init.normal_(self.item_id_emb.weight, std=0.1)

        # Text projection: LLM_DIM → embed_dim
        self.text_prj = nn.Linear(llm_dim, embed_dim)

        # Gate MLP: concat [E_graph || z_LLM] → scalar logit
        # Phi tuyến tính thay vì cộng tuyến tính — học tương tác giữa hai nguồn
        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1)
        )
        # Khởi tạo nhỏ để gate bắt đầu gần sigmoid(0) = 0.5
        for m in self.gate_mlp.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.001)
                nn.init.zeros_(m.bias)

        # Scalar learnable: tín hiệu tần suất & cosine similarity
        self.w_freq  = nn.Parameter(torch.tensor(0.5))
        self.w_sim   = nn.Parameter(torch.tensor(0.5))
        # Tín hiệu niche cho user: giảm gamma khi user thích hàng ngách nhiều
        self.w_niche = nn.Parameter(torch.tensor(0.3))

        # Alpha global: kết hợp z_G và z_L ở output
        self.alpha = nn.Parameter(torch.tensor(0.5))

    # ── Tính gate γ^(l) — đối xứng user-item ─────────────────────────────────
    def _compute_gate(self, x_graph, z_llm, freq_all, niche_ratio_all=None):
        """
        x_graph    : [N_total × d] embedding graph tại tầng l
        z_llm      : [N_total × d] LLM embedding đã project (float32)
        freq_all   : [N_total] frequency (user_freq || item_freq ghép lại)
        niche_ratio_all: [N_total] chỉ phần user có giá trị, phần item = 0
        Trả về gamma: [N_total × 1]
        """
        # MLP phi tuyến
        gate_input = torch.cat([x_graph, z_llm], dim=1)  # [N, 2d]
        base       = self.gate_mlp(gate_input)             # [N, 1]

        # Frequency signal (đối xứng: áp dụng cả user lẫn item)
        freq_signal = torch.log1p(freq_all).unsqueeze(1) * self.w_freq  # [N, 1]

        # Cosine similarity signal
        cos_sim    = F.cosine_similarity(x_graph, z_llm, dim=1, eps=1e-8).unsqueeze(1)
        cos_signal = cos_sim * self.w_sim                  # [N, 1]

        gate_logit = base + freq_signal + cos_signal       # [N, 1]

        # Niche ratio: chỉ áp dụng cho phần user (index 0..N_users-1)
        if niche_ratio_all is not None:
            N_u = self.num_users
            niche_signal = niche_ratio_all[:N_u].unsqueeze(1) * self.w_niche
            gate_logit[:N_u] = gate_logit[:N_u] - niche_signal

        gamma = torch.sigmoid(gate_logit)  # [N, 1]

        # Hard masking: cold nodes (freq=0) → gamma=0 → 100% LLM
        cold_mask = (freq_all == 0).unsqueeze(1)
        gamma     = torch.where(cold_mask, torch.zeros_like(gamma), gamma)

        return gamma

    # ── Intra-layer gated forward — V2 core change ────────────────────────────
    def forward_gcn_gated(self, adj, z_llm_all, freq_all, niche_ratio_all=None):
        """
        Thực hiện intra-layer gated fusion theo RecMind Eq.6-8:
          Ê^(l) = γ^(l)·E^(l) + (1-γ^(l))·z_LLM
          E^(l+1) = Â·Ê^(l)
        γ được tính riêng mỗi tầng vì E^(l) thay đổi theo tầng.
        """
        # Base embedding (float32, trainable)
        x = torch.cat([self.user_id_emb.weight,
                        self.item_id_emb.weight], dim=0)  # [N_total, d]
        embs = [x]

        for _ in range(self.gcn_layers):
            # Tính gate tại tầng này — dùng x hiện tại (thay đổi mỗi tầng)
            gamma   = self._compute_gate(x, z_llm_all, freq_all, niche_ratio_all)

            # Tối ưu in-place thay vì tạo 3 tensor tạm: gamma*x + (1-gamma)*z_llm
            # x_fused = gamma * x + (1-gamma) * z_llm_all
            # Dùng addcmul_ an toàn hơn khi float mix
            x_fused = torch.addcmul(z_llm_all, gamma, x - z_llm_all)  # [N, d]

            # LightGCN propagation với fused state (phải cùng dtype với adj = float16)
            x = torch.sparse.mm(adj, x_fused.to(adj.dtype)).to(torch.float32)
            embs.append(x)

        # Layer-averaged output (Eq.2 LightGCN)
        z_G = sum(embs) / len(embs)
        return z_G

    # ── Text projection ──────────────────────────────────────────────────────
    def project_text(self, llm_emb):
        return self.text_prj(llm_emb)

    # ── Final representation (Eq.9 RecMind) ─────────────────────────────────
    def get_final_repr(self, z_G, z_L):
        alpha = torch.sigmoid(self.alpha)
        return alpha * z_G + (1.0 - alpha) * z_L

    # ── Forward pass ─────────────────────────────────────────────────────────
    def forward(self, adj, z_llm_all, freq_all,
                batch_users, batch_pos, batch_neg,
                u_llm, pos_llm, neg_llm,
                niche_ratio_all=None):
        """
        z_llm_all : [N_total, d] đã project — được cache ngoài forward để tiết kiệm
        freq_all  : [N_total] — user_freq || item_freq
        """
        # Intra-layer GCN với gated fusion
        z_G_all = self.forward_gcn_gated(adj, z_llm_all, freq_all, niche_ratio_all)
        user_G, item_G = z_G_all.split([self.num_users, self.num_items])

        # Lấy embedding theo batch index
        u_G   = user_G[batch_users]
        pos_G = item_G[batch_pos]
        neg_G = item_G[batch_neg]

        # Project LLM (batch)
        u_L   = self.project_text(u_llm)
        pos_L = self.project_text(pos_llm)
        neg_L = self.project_text(neg_llm)

        # Final repr
        final_u   = self.get_final_repr(u_G,   u_L)
        final_pos = self.get_final_repr(pos_G, pos_L)
        final_neg = self.get_final_repr(neg_G, neg_L)

        # Trả thêm z_G, z_L cho alignment loss
        return final_u, final_pos, final_neg, u_G, u_L, pos_G, pos_L

print("[INFO] TARecMindV2 class đã định nghĩa.")

[INFO] TARecMindV2 class đã định nghĩa.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: HÀM MẤT MÁT
# ═══════════════════════════════════════════════════════════════════════════════

def bpr_loss(user_emb, pos_emb, neg_emb):
    """BPR loss chuẩn với L2-normalized embeddings."""
    pos_scores = (user_emb * pos_emb).sum(dim=1)
    neg_scores = (user_emb * neg_emb).sum(dim=1)
    return -F.logsigmoid(pos_scores - neg_scores).mean()


def lagcl_noise(embeddings, item_freq, noise_scale=0.08):
    """LAGCL: noise nhỏ hơn với tail items để augmentation hiệu quả hơn."""
    noise_weight = noise_scale / torch.log(item_freq + 2.0)
    return embeddings + torch.randn_like(embeddings) * noise_weight.unsqueeze(1)


def contrastive_loss_lagcl(v1, v2, temperature=0.15):
    v1 = F.normalize(v1, dim=1)
    v2 = F.normalize(v2, dim=1)
    sim    = torch.mm(v1, v2.t()) / temperature
    labels = torch.arange(sim.size(0), device=v1.device)
    return F.cross_entropy(sim, labels)


def tail_weighted_alignment_loss(z_graph, z_text, freq, temperature=0.15, subbatch=512):
    """
    InfoNCE alignment với trọng số tail: w = 1/log(1+freq)
    Tính theo sub-batch để tránh OOM với N lớn.
    """
    z_g = F.normalize(z_graph, dim=1)
    z_t = F.normalize(z_text,  dim=1)
    N   = z_g.size(0)
    w   = 1.0 / torch.log(1.0 + freq + 1e-8)
    labels_full = torch.arange(N, device=z_g.device)
    total_loss  = torch.tensor(0.0, device=z_g.device)

    for start in range(0, N, subbatch):
        end       = min(start + subbatch, N)
        sim_chunk = torch.mm(z_g[start:end], z_t.t()) / temperature
        ce_chunk  = F.cross_entropy(sim_chunk, labels_full[start:end], reduction="none")
        total_loss = total_loss + (w[start:end] * ce_chunk).sum()

    return total_loss / (N * w.mean() + 1e-8)

print("[INFO] Các hàm loss đã định nghĩa.")

[INFO] Các hàm loss đã định nghĩa.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8: NEGATIVE SAMPLING 3-THÀNH-PHẦN
#
# Thành phần 1 (40%): Uniform — giữ phân phối tự nhiên
# Thành phần 2 (40%): Popularity-biased (freq^0.75) — buộc model phân biệt head
# Thành phần 3 (20%): Tail-edge oversampling — đảm bảo gradient từ tail items
# ═══════════════════════════════════════════════════════════════════════════════
def sample_negatives_3component(batch_size, num_items, neg_prob_t,
                                  tail_edge_indices, train_edge_index, device):
    """
    Trả về tensor neg_items [batch_size] theo chiến lược 3 thành phần.
    """
    n1 = int(batch_size * 0.4)   # Uniform
    n2 = int(batch_size * 0.4)   # Popularity-biased
    n3 = batch_size - n1 - n2    # Tail oversampled

    # Thành phần 1: Uniform
    neg1 = torch.randint(0, num_items, (n1,), device=device)

    # Thành phần 2: Popularity-biased (freq^0.75 đã precompute trong neg_prob_t)
    neg2 = torch.multinomial(neg_prob_t, n2, replacement=True)

    # Thành phần 3: Lấy item từ các tail edges (item có freq ≤ 5)
    # → đảm bảo model cũng học phân biệt negative trong vùng tail
    tail_sample_idx = tail_edge_indices[
        torch.randint(0, len(tail_edge_indices), (n3,), device=device)
    ]
    neg3 = train_edge_index[1, tail_sample_idx]

    return torch.cat([neg1, neg2, neg3])

print("[INFO] Hàm negative sampling 3-thành-phần đã sẵn sàng.")

[INFO] Hàm negative sampling 3-thành-phần đã sẵn sàng.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9: STRATIFIED EVALUATION — 1000 USERS PHÂN TẦNG
#
# Evaluation trong training loop: 1000 users phân tầng thay vì 100 random
# → phát hiện regression head/tail ngay từng epoch
# ═══════════════════════════════════════════════════════════════════════════════
def build_stratified_eval_sample(val_edges, user_activity_group_t,
                                  user_niche_t=None, n=1000, seed=42):
    """
    Xây dựng 4 nhóm mỗi nhóm n//4 users từ val_edges:
      - SUPER_ACTIVE (activity=2)
      - ACTIVE (activity=1)
      - INACTIVE (activity=0)
      - NICHE_LOVERS (niche_ratio > 0.5 nếu có, ngược lại dùng INACTIVE bổ sung)
    Lưu vào dict để tái sử dụng mỗi lần eval.
    """
    torch.manual_seed(seed)
    per_group = n // 4
    val_u = val_edges[0].cpu()
    val_i = val_edges[1].cpu()
    act_g = user_activity_group_t.cpu()

    def sample_group(mask_fn, k):
        mask    = mask_fn(val_u)
        indices = mask.nonzero().squeeze()
        if indices.dim() == 0: indices = indices.unsqueeze(0)
        if len(indices) == 0:  return None, None
        chosen = indices[torch.randperm(len(indices))[:k]]
        return val_u[chosen], val_i[chosen]

    groups = {}
    groups["SUPER_ACTIVE"] = sample_group(lambda u: act_g[u] == 2, per_group)
    groups["ACTIVE"]       = sample_group(lambda u: act_g[u] == 1, per_group)
    groups["INACTIVE"]     = sample_group(lambda u: act_g[u] == 0, per_group)

    if user_niche_t is not None:
        niche_cpu = user_niche_t.cpu()
        groups["NICHE_LOVERS"] = sample_group(lambda u: niche_cpu[u] > 0.5, per_group)
    else:
        groups["NICHE_LOVERS"] = groups["INACTIVE"]

    return groups


def evaluate_stratified(model, adj, z_llm_all_cached, freq_all,
                         user_emb_llm, item_emb_llm,
                         eval_groups, item_train_freq_t,
                         num_users, num_items, K=20, device="cuda"):
    """
    Full-ranking evaluation trên 1000 users phân tầng.
    Dùng z_llm_all_cached đã precomputed để tiết kiệm compute.
    """
    model.eval()

    # Ngưỡng phân loại item
    freq_np    = item_train_freq_t.cpu().numpy()
    sorted_f   = np.sort(freq_np)[::-1]
    pareto_idx = np.searchsorted(np.cumsum(sorted_f), 0.8 * sorted_f.sum())
    HEAD_THRESH = max(int(sorted_f[min(pareto_idx, len(sorted_f)-1)]), 6)
    TAIL_THRESH = 5

    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda")):
        # Tính toàn bộ item representations một lần
        z_G_all = model.forward_gcn_gated(adj, z_llm_all_cached, freq_all)
        _, item_G_all = z_G_all.split([num_users, num_items])

        # Project item LLM theo chunk để tránh OOM
        item_L_chunks = []
        for s in range(0, num_items, 32768):
            e = min(s + 32768, num_items)
            item_L_chunks.append(model.project_text(item_emb_llm[s:e].to(device)))
        item_L_all = torch.cat(item_L_chunks, dim=0)

        item_final = F.normalize(
            model.get_final_repr(item_G_all, item_L_all), p=2, dim=1
        )

    results_by_group = {}
    overall_hit = overall_ndcg = overall_n = 0

    # Phân tầng kết quả theo nhóm item
    item_category = torch.zeros(num_items, dtype=torch.long, device=device)
    item_category[item_train_freq_t == 0]          = 0  # cold
    item_category[(item_train_freq_t > 0) &
                  (item_train_freq_t <= TAIL_THRESH)] = 1  # tail
    item_category[item_train_freq_t >= HEAD_THRESH] = 2  # head

    for gname, (g_users, g_items) in eval_groups.items():
        if g_users is None:
            results_by_group[gname] = {"Recall": 0.0, "NDCG": 0.0, "n": 0}
            continue

        g_u   = g_users.to(device)
        g_i   = g_items.to(device)
        hit_g = ndcg_g = 0.0

        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda")):
            for idx in range(len(g_u)):
                u_idx    = g_u[idx].item()
                pos_item = g_i[idx].item()

                # Full-ranking: score với toàn bộ catalog
                u_llm_v  = user_emb_llm[u_idx:u_idx+1].to(device)
                u_G_v    = z_G_all[u_idx:u_idx+1]
                u_L_v    = model.project_text(u_llm_v)
                u_final  = F.normalize(model.get_final_repr(u_G_v, u_L_v), p=2, dim=1)

                scores   = (u_final * item_final).sum(dim=1)
                rank     = (scores > scores[pos_item]).sum().item() + 1

                hit_g  += 1 if rank <= K else 0
                ndcg_g += (1.0 / np.log2(rank + 1)) if rank <= K else 0.0

        n_g = len(g_u)
        results_by_group[gname] = {
            "Recall": hit_g / n_g,
            "NDCG":   ndcg_g / n_g,
            "n":      n_g
        }
        overall_hit  += hit_g
        overall_ndcg += ndcg_g
        overall_n    += n_g

    results_by_group["OVERALL"] = {
        "Recall": overall_hit / max(overall_n, 1),
        "NDCG":   overall_ndcg / max(overall_n, 1),
        "n":      overall_n
    }
    return results_by_group


def print_stratified_results(epoch, results):
    print(f"\n[VAL STRATIFIED] Epoch {epoch:02d}")
    print(f"  {'Nhóm':<16} | {'Recall@20':>10} | {'NDCG@20':>10} | {'N':>6}")
    print(f"  {'-'*50}")
    for gname, v in results.items():
        print(f"  {gname:<16} | {v['Recall']:>10.4f} | {v['NDCG']:>10.4f} | {v['n']:>6}")

print("[INFO] Stratified evaluation functions đã sẵn sàng.")

[INFO] Stratified evaluation functions đã sẵn sàng.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10: CHECKPOINT MANAGER V2
# Bổ sung: lưu z_llm_projected.pt mỗi 5 epoch để tránh recompute khi resume
# ═══════════════════════════════════════════════════════════════════════════════
class ColabCheckpointManager:
    def __init__(self, save_dir):
        self.save_dir   = save_dir
        self.best_path  = os.path.join(save_dir, "tarecmind_best.pth")
        self.last_path  = os.path.join(save_dir, "tarecmind_last.pth")
        self.hist_path  = os.path.join(save_dir, "training_history.json")
        self.proj_path  = os.path.join(save_dir, "z_llm_projected.pt")
        self.eval_path  = os.path.join(save_dir, "eval_sample_indices.pt")

    def save(self, model, optimizer, epoch, metrics, history_list, cfg, is_best=False):
        payload = {
            "epoch":                epoch,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "metrics":              metrics,
            "config":               {k: v for k, v in cfg.items()
                                     if not k.startswith("_")},
        }
        torch.save(payload, self.last_path)
        if is_best:
            torch.save(payload, self.best_path)
            print(f"[CKPT] New BEST checkpoint saved (Recall@20={metrics.get('Recall@20_overall', 0):.4f})")

        # Gộp lịch sử không ghi đè
        current = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f:
                    current = json.load(f)
            except: pass
        existing_epochs = {h["epoch"] for h in current}
        for entry in history_list:
            if entry["epoch"] not in existing_epochs:
                current.append(entry)
        current.sort(key=lambda x: x["epoch"])
        with open(self.hist_path, "w") as f:
            json.dump(current, f, indent=2)

    def save_projected_llm(self, z_llm_projected):
        """Lưu cached projection lên Drive (gọi mỗi 5 epoch)."""
        torch.save(z_llm_projected.cpu().half(), self.proj_path)
        print(f"[CKPT] z_llm_projected saved to Drive ({z_llm_projected.shape}).")

    def load_projected_llm(self, device):
        """Tải cached projection từ Drive nếu có."""
        if os.path.exists(self.proj_path):
            z = torch.load(self.proj_path, map_location=device, weights_only=True)
            return z.float()
        return None

    def try_resume(self, model, optimizer):
        if not os.path.exists(self.last_path):
            print("[CKPT] Không tìm thấy checkpoint. Bắt đầu từ epoch 1.")
            return 0, 0.0, []

        print(f"[CKPT] Tìm thấy checkpoint: {self.last_path}")
        ckpt = torch.load(self.last_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"])
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception as e:
            print(f"[WARN] Không load optimizer state: {e}")

        start_epoch = ckpt["epoch"]
        history = []
        if os.path.exists(self.hist_path):
            try:
                with open(self.hist_path, "r") as f:
                    history = json.load(f)
            except: pass

        best_recall = max((h.get("Recall@20_overall", 0.0) for h in history), default=0.0)
        print(f"[CKPT] Resume epoch {start_epoch+1} | Best Recall: {best_recall:.4f}")
        return start_epoch, best_recall, history

print("[INFO] ColabCheckpointManager đã sẵn sàng.")

[INFO] ColabCheckpointManager đã sẵn sàng.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11: KHỞI TẠO MÔ HÌNH & PRECOMPUTE z_llm_projected
#
# z_llm_all = text_prj(cat(user_llm || item_llm)):
#   - Tính một lần mỗi 5 epoch thay vì mỗi step → tiết kiệm đáng kể VRAM & compute
#   - Cache lên Drive để khôi phục khi resume
# ═══════════════════════════════════════════════════════════════════════════════
# Dọn dẹp VRAM
for var in ["model", "optimizer", "z_llm_all_cached", "item_final"]:
    if var in globals():
        del globals()[var]
gc.collect(); torch.cuda.empty_cache()

# ── Khởi tạo model ──────────────────────────────────────────────────────────
model = TARecMindV2(
    num_users  = num_users,
    num_items  = num_items,
    embed_dim  = CFG["EMBED_DIM"],
    llm_dim    = CFG["LLM_DIM"],
    gcn_layers = CFG["GCN_LAYERS"],
).to(device)

scaler   = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
ckpt_mgr = ColabCheckpointManager(CFG["SAVE_DIR"])

# Optimizer tạm để resume
optimizer_init = optim.Adam(model.parameters(), lr=CFG["LR_JOINT"])
resume_epoch, best_recall, history = ckpt_mgr.try_resume(model, optimizer_init)

# ── Precompute z_llm_all_cached ──────────────────────────────────────────────
# freq_all = cat(user_freq, item_freq) — dùng cho gate
freq_all_cpu = torch.cat([
    user_train_freq_t.cpu(),
    item_train_freq_t.cpu()
]).float()
freq_all = freq_all_cpu.to(device)

def precompute_z_llm(model, user_emb_llm, item_emb_llm, device, chunk=65536):
    """
    Project toàn bộ LLM embeddings qua text_prj một lần.
    Tính theo chunk để tránh OOM.
    Kết quả float32 trên GPU.
    """
    model.eval()
    all_emb = torch.cat([user_emb_llm.float(), item_emb_llm.float()], dim=0)
    N       = all_emb.shape[0]
    parts   = []
    with torch.no_grad():
        for s in range(0, N, chunk):
            e     = min(s + chunk, N)
            chunk_ = all_emb[s:e].to(device)
            parts.append(model.project_text(chunk_).cpu())
    return torch.cat(parts, dim=0).to(device)

# Thử load từ Drive trước, nếu không có thì tính mới
z_llm_all_cached = ckpt_mgr.load_projected_llm(device)
if z_llm_all_cached is None:
    print("[INFO] Tính z_llm_all_cached từ đầu...")
    z_llm_all_cached = precompute_z_llm(model, user_emb_llm, item_emb_llm, device)
    ckpt_mgr.save_projected_llm(z_llm_all_cached)
else:
    print(f"[INFO] Đã tải z_llm_all_cached từ Drive: {z_llm_all_cached.shape}")

# Hash để phát hiện khi text_prj thay đổi
_last_proj_hash = hash(model.text_prj.weight.data.cpu().numpy().tobytes())
_proj_refresh_every = 5   # refresh mỗi 5 epoch

print(f"[READY] Model {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"[READY] z_llm_all_cached: {z_llm_all_cached.shape} | freq_all: {freq_all.shape}")
gc.collect()

[CKPT] Tìm thấy checkpoint: /content/drive/MyDrive/tarecmind/weights/tarecmind_last.pth
[WARN] Không load optimizer state: loaded state dict has a different number of parameter groups
[CKPT] Resume epoch 43 | Best Recall: 0.0000
[INFO] Đã tải z_llm_all_cached từ Drive: torch.Size([3093932, 128])
[READY] Model 396,089,093 parameters
[READY] z_llm_all_cached: torch.Size([3093932, 128]) | freq_all: torch.Size([3093932])


18

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12: CHUẨN BỊ EVAL SAMPLE & USER NICHE RATIO
# Tính một lần, lưu vào Drive để tái sử dụng
# ═══════════════════════════════════════════════════════════════════════════════
# ── Load user activity group ─────────────────────────────────────────────────
user_activity_npy = np.load(download_hf("gold/gold_user_activity_group.npy"))
user_activity_t   = torch.from_numpy(user_activity_npy).long().to(device)

# ── User niche ratio: tỷ lệ tương tác với tail items ────────────────────────
# niche_ratio[u] = số tail interactions của u / tổng interactions của u
niche_path = os.path.join(CFG["DATA_DIR"], "user_niche_ratio.pt")

if os.path.exists(niche_path):
    user_niche_t = torch.load(niche_path, map_location=device, weights_only=True)
    print(f"[INFO] Đã tải user_niche_ratio từ Drive: {user_niche_t.shape}")
else:
    print("[INFO] Tính user_niche_ratio từ train_edge_index...")
    users_all    = train_edge_index[0]       # [E]
    items_all    = train_edge_index[1]       # [E]
    is_tail_item = item_train_freq_t[items_all] <= 5  # [E] bool

    user_tail_cnt  = torch.bincount(users_all[is_tail_item],  minlength=num_users).float()
    user_total_cnt = torch.bincount(users_all,                minlength=num_users).float()
    user_niche_t   = user_tail_cnt / torch.clamp(user_total_cnt, min=1.0)

    # Smooth: users với ít tương tác dùng global mean
    global_mean  = user_niche_t[user_total_cnt >= 2].mean()
    user_niche_t = torch.where(user_total_cnt < 2, global_mean, user_niche_t)

    torch.save(user_niche_t, niche_path)
    print(f"[INFO] Đã lưu user_niche_ratio. Mean={user_niche_t.mean():.4f}")

# ── Niche ratio cho toàn bộ nodes (user || item padding zeros) ───────────────
niche_ratio_all = torch.cat([
    user_niche_t,
    torch.zeros(num_items, device=device)
])

# ── Build stratified eval sample (1000 users, tính một lần) ─────────────────
eval_groups_path = os.path.join(CFG["DATA_DIR"], "eval_groups.pt")

if os.path.exists(eval_groups_path):
    eval_groups = torch.load(eval_groups_path, map_location="cpu", weights_only=True)
    print(f"[INFO] Đã tải eval_groups từ Drive.")
    # Chuyển tuple từ dict về dạng chuẩn
    eval_groups = {k: (v[0], v[1]) for k, v in eval_groups.items()}
else:
    eval_groups = build_stratified_eval_sample(
        val_edges_t, user_activity_t.cpu(),
        user_niche_t.cpu(), n=CFG["EVAL_STRAT_N"]
    )
    # Lưu dạng dict tensor để torch.save hoạt động
    save_groups = {}
    for k, (u, i) in eval_groups.items():
        if u is not None:
            save_groups[k] = (u, i)
    torch.save(save_groups, eval_groups_path)
    print(f"[INFO] Đã lưu eval_groups lên Drive.")

for gname, (g_u, g_i) in eval_groups.items():
    if g_u is not None:
        print(f"  {gname}: {len(g_u)} users")

del user_activity_npy; gc.collect()

[INFO] Đang tải gold/gold_user_activity_group.npy...


gold/gold_user_activity_group.npy:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

[INFO] Đã tải user_niche_ratio từ Drive: torch.Size([1483920])
[INFO] Đã tải eval_groups từ Drive.
  SUPER_ACTIVE: 250 users
  ACTIVE: 250 users
  INACTIVE: 250 users
  NICHE_LOVERS: 62 users


116

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 13: TRAINING LOOP V2 — INTRA-LAYER GATED FUSION
# ═══════════════════════════════════════════════════════════════════════════════
patience_cnt = 0
best_epoch   = resume_epoch

# ── Optimizer với differential LR ───────────────────────────────────────────
joint_params = [
    # GNN embeddings: học với LR tiêu chuẩn
    {"params": model.user_id_emb.parameters(), "lr": CFG["LR_JOINT"]},
    {"params": model.item_id_emb.parameters(), "lr": CFG["LR_JOINT"]},
    # Text projection & gate: học chậm hơn để bảo vệ semantic space
    # V2: LR × 0.05 thay vì 0.1 vì gate MLP phức tạp hơn
    {"params": model.text_prj.parameters(),  "lr": CFG["LR_JOINT"] * 0.05},
    {"params": model.gate_mlp.parameters(),  "lr": CFG["LR_JOINT"] * 0.05},
    {"params": [model.w_freq, model.w_sim, model.w_niche, model.alpha],
               "lr": CFG["LR_JOINT"] * 0.05},
]
optimizer = optim.Adam(joint_params, weight_decay=CFG["WEIGHT_DECAY"])

if resume_epoch > 0:
    try: optimizer.load_state_dict(optimizer_init.state_dict())
    except: pass

for group in optimizer.param_groups:
    group.setdefault("initial_lr", group.get("lr", CFG["LR_JOINT"]))

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["EPOCHS"], eta_min=1e-5, last_epoch=resume_epoch - 1
)

print(f"\n--- BẮT ĐẦU TRAINING V2 (Epoch {resume_epoch+1}/{CFG['EPOCHS']}) ---")
print(f"[INFO] Batch={CFG['BATCH_SIZE']} | Accum={CFG['ACCUM_STEPS']}")

# ── Vòng lặp chính ──────────────────────────────────────────────────────────
for epoch in range(resume_epoch, CFG["EPOCHS"]):
    t_epoch_start = time.time()
    model.train()

    total_loss = total_bpr = total_align = total_cl = 0.0
    steps_per_epoch = n_train_edges // CFG["BATCH_SIZE"]

    # ── Refresh z_llm_all_cached mỗi _proj_refresh_every epoch ───────────────
    if epoch > 0 and epoch % _proj_refresh_every == 0:
        new_hash = hash(model.text_prj.weight.data.cpu().numpy().tobytes())
        if new_hash != _last_proj_hash:
            print(f"[INFO] Refreshing z_llm_projected (epoch {epoch+1})...")
            model.eval()
            z_llm_all_cached = precompute_z_llm(model, user_emb_llm, item_emb_llm, device)
            ckpt_mgr.save_projected_llm(z_llm_all_cached)
            _last_proj_hash = new_hash
            model.train()

    # ── Sampling sizes ────────────────────────────────────────────────────────
    # 40% tail-oversampled edges + 60% mixed negative batch
    half_tail = CFG["BATCH_SIZE"] // 2

    with tqdm(total=steps_per_epoch * CFG["BATCH_SIZE"],
              desc=f"Ep {epoch+1:02d}/{CFG['EPOCHS']}", ncols=110) as pbar:
        optimizer.zero_grad(set_to_none=True)

        for step in range(steps_per_epoch):
            # ── 1. POSITIVE SAMPLING: 50% uniform + 50% tail ─────────────────
            std_idx  = torch.randint(0, n_train_edges, (half_tail,), device=device)
            tail_idx = tail_edge_indices[
                torch.randint(0, len(tail_edge_indices), (half_tail,), device=device)
            ]
            batch_idx   = torch.cat([std_idx, tail_idx])
            batch_edges = train_edge_index[:, batch_idx]
            users       = batch_edges[0]
            pos_items   = batch_edges[1]

            # ── 2. NEGATIVE SAMPLING 3-THÀNH-PHẦN ────────────────────────────
            neg_items = sample_negatives_3component(
                len(users), num_items, neg_prob_t,
                tail_edge_indices, train_edge_index, device
            )

            # ── 3. Lấy LLM embeddings theo batch ─────────────────────────────
            users_cpu     = users.cpu()
            pos_items_cpu = pos_items.cpu()
            neg_items_cpu = neg_items.cpu()

            u_llm   = user_emb_llm[users_cpu].to(device, non_blocking=True)
            pos_llm = item_emb_llm[pos_items_cpu].to(device, non_blocking=True)
            neg_llm = item_emb_llm[neg_items_cpu].to(device, non_blocking=True)

            pos_freq_batch = item_train_freq_t[pos_items]
            neg_freq_batch = item_train_freq_t[neg_items]

            # ── 4. FORWARD PASS ───────────────────────────────────────────────
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                final_u, final_pos, final_neg, u_G, u_L, pos_G, pos_L = model(
                    sparse_adj,
                    z_llm_all_cached,  # precomputed, không tính lại mỗi step
                    freq_all,
                    users, pos_items, neg_items,
                    u_llm, pos_llm, neg_llm,
                    niche_ratio_all=niche_ratio_all
                )

                # L2-normalize trước BPR (đồng bộ với cosine eval)
                final_u   = F.normalize(final_u,   p=2, dim=1)
                final_pos = F.normalize(final_pos, p=2, dim=1)
                final_neg = F.normalize(final_neg, p=2, dim=1)

                loss_bpr = bpr_loss(final_u, final_pos, final_neg)

                loss_align = (
                    tail_weighted_alignment_loss(
                        u_G, u_L, pos_freq_batch,
                        CFG["TEMPERATURE"], CFG["ALIGN_SUBBATCH"]
                    )
                    + tail_weighted_alignment_loss(
                        pos_G, pos_L, pos_freq_batch,
                        CFG["TEMPERATURE"], CFG["ALIGN_SUBBATCH"]
                    )
                )

                loss_cl = contrastive_loss_lagcl(
                    lagcl_noise(final_pos, pos_freq_batch, CFG["NOISE_SCALE"]),
                    lagcl_noise(final_pos, pos_freq_batch, CFG["NOISE_SCALE"]),
                    CFG["TEMPERATURE"]
                )

                loss = (
                    loss_bpr
                    + CFG["LAMBDA_ALIGN"] * loss_align
                    + CFG["LAMBDA_CL"]    * loss_cl
                ) / CFG["ACCUM_STEPS"]

            # ── 5. BACKWARD ───────────────────────────────────────────────────
            scaler.scale(loss).backward()

            if (step + 1) % CFG["ACCUM_STEPS"] == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            n_batch      = len(users)
            total_loss  += loss.item() * CFG["ACCUM_STEPS"] * n_batch
            total_bpr   += loss_bpr.item()    * n_batch
            total_align += loss_align.item()  * n_batch
            total_cl    += loss_cl.item()     * n_batch

            pbar.update(n_batch)
            pbar.set_postfix({
                "bpr":   f"{loss_bpr.item():.3f}",
                "align": f"{loss_align.item():.3f}",
                "cl":    f"{loss_cl.item():.3f}"
            })

        # Flush gradient accumulation ở cuối epoch
        if steps_per_epoch % CFG["ACCUM_STEPS"] != 0:
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)

    scheduler.step()
    t_epoch = time.time() - t_epoch_start
    n_proc  = steps_per_epoch * CFG["BATCH_SIZE"]
    avg_loss = total_loss / n_proc

    print(f"[TRAIN] Ep {epoch+1:02d} | {t_epoch:.1f}s | "
          f"Loss={avg_loss:.4f} BPR={total_bpr/n_proc:.4f} "
          f"Align={total_align/n_proc:.4f} CL={total_cl/n_proc:.4f}")

    # ── 6. EVALUATION mỗi EVAL_EVERY epoch ───────────────────────────────────
    metrics = {}
    is_best = False

    if (epoch + 1) % CFG["EVAL_EVERY"] == 0:
        t0 = time.time()
        eval_results = evaluate_stratified(
            model, sparse_adj, z_llm_all_cached, freq_all,
            user_emb_llm, item_emb_llm,
            eval_groups, item_train_freq_t,
            num_users, num_items, K=20, device=device
        )
        t_eval = time.time() - t0
        print_stratified_results(epoch + 1, eval_results)
        print(f"[EVAL] Time: {t_eval:.1f}s")

        # Detect head regression
        if len(history) >= 10:
            past5_head = [h.get("Recall@20_HEAD", 0) for h in history[-5:]]
            curr_head  = eval_results.get("SUPER_ACTIVE", {}).get("Recall", 0)
            if past5_head and curr_head < 0.95 * max(past5_head):
                print(f"[WARN] HEAD REGRESSION DETECTED (curr={curr_head:.4f})")

        # Flatten metrics để lưu history
        for gname, v in eval_results.items():
            metrics[f"Recall@20_{gname}"] = v["Recall"]
            metrics[f"NDCG@20_{gname}"]   = v["NDCG"]
        metrics["epoch"] = epoch + 1
        metrics["loss"]  = avg_loss

        r_all = eval_results.get("OVERALL", {}).get("Recall", 0.0)
        if r_all > best_recall:
            best_recall  = r_all
            best_epoch   = epoch + 1
            patience_cnt = 0
            is_best      = True
            print(f"✅ NEW BEST! Recall@20={best_recall:.4f}")
        else:
            patience_cnt += 1
            print(f"   Patience {patience_cnt}/{CFG['PATIENCE']}")

        history.append(metrics)

    # ── 7. CHECKPOINT ─────────────────────────────────────────────────────────
    if CFG["SAVE_EVERY_EPOCH"]:
        ckpt_mgr.save(model, optimizer, epoch + 1, metrics, history, CFG, is_best)

    if patience_cnt >= CFG["PATIENCE"]:
        print(f"\n[INFO] Early stopping tại epoch {epoch+1}.")
        break

    if device == "cuda":
        torch.cuda.empty_cache()

# Lưu lần cuối
_keepalive_stop.set()
ckpt_mgr.save(model, optimizer, epoch + 1, metrics, history, CFG, is_best=False)
print(f"\n✅ HOÀN TẤT V2! Best Recall@20={best_recall:.4f} (Ep {best_epoch})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 14: INFERENCE PREPARATION (ID MAPPING, MASKING & SEGMENTATION FLAGS)
# ═══════════════════════════════════════════════════════════════════════════════
!pip install -q huggingface_hub fsspec pyarrow

import pandas as pd
from collections import defaultdict
import os

print("\n--- [BƯỚC 5] ID MAPPING: CHUẨN BỊ TẬP TEST ---")

# Tải best model nếu có
if os.path.exists(ckpt_mgr.best_path):
    ckpt = torch.load(ckpt_mgr.best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[INFO] Đã load best model (epoch {ckpt['epoch']}).")

df_test = load_dataset("chuongdo1104/amazon-2023-bronze",
                        data_dir="bronze/bronze_test.parquet",
                        split="train").to_pandas()

# Nhận diện linh hoạt tên cột
user_int_col = "user_idx" if "user_idx" in df_test.columns else "user_id" if "user_id" in df_test.columns else "reviewer_id"
item_int_col = "item_idx" if "item_idx" in df_test.columns else "item_id" if "item_id" in df_test.columns else "parent_asin"

if df_test[user_int_col].dtype == object:
    df_user_map = pd.read_parquet(download_hf("gold/gold_user_id_map.parquet", local_dir=CFG["CACHE_DIR"]))
    df_item_map = pd.read_parquet(download_hf("gold/gold_item_id_map.parquet", local_dir=CFG["CACHE_DIR"]))
    u_map = dict(zip(df_user_map["reviewer_id"], df_user_map["user_idx"]))
    i_map = dict(zip(df_item_map["parent_asin"], df_item_map["item_idx"]))
    df_test["user_idx"] = df_test[user_int_col].map(u_map)
    df_test["item_idx"] = df_test[item_int_col].map(i_map)
    df_test = df_test.dropna(subset=["user_idx", "item_idx"])
    test_users = torch.tensor(df_test["user_idx"].astype(int).values, dtype=torch.long)
    test_items = torch.tensor(df_test["item_idx"].astype(int).values, dtype=torch.long)
else:
    test_users = torch.tensor(df_test[user_int_col].values, dtype=torch.long)
    test_items = torch.tensor(df_test[item_int_col].values, dtype=torch.long)

mask              = (test_items < num_items) & (test_users < num_users)
test_users_filt   = test_users[mask].to(device)
test_items_filt   = test_items[mask].to(device)
print(f"[INFO] Test interactions hợp lệ: {len(test_users_filt):,}")

print("\n--- [BƯỚC 1] TRAINING ITEM MASKING (LỌC LỊCH SỬ) ---")
train_user_dict = defaultdict(list)
for u, i in zip(train_edge_index[0].cpu().tolist(), train_edge_index[1].cpu().tolist()):
    train_user_dict[u].append(i)
for u in train_user_dict:
    train_user_dict[u] = torch.tensor(train_user_dict[u], dtype=torch.long, device=device)
print(f"[INFO] Đã tạo Masking Dictionary cho {len(train_user_dict):,} users.")

print("\n--- [BƯỚC 3] SEGMENTATION FLAGGING (CHUẨN BỊ NHÃN PHÂN TẦNG) ---")
# Phân tầng Item (0=cold, 1=tail, 2=head)
freq_np = item_train_freq_t.cpu().numpy()
sorted_f = np.sort(freq_np)[::-1]
pareto_idx = np.searchsorted(np.cumsum(sorted_f), 0.8 * sorted_f.sum())
HEAD_THRESH = max(int(sorted_f[min(pareto_idx, len(sorted_f)-1)]), 6)

item_groups_eval = torch.zeros(num_items, dtype=torch.long, device=device)
item_groups_eval[(item_train_freq_t > 0) & (item_train_freq_t <= 5)] = 1   # TAIL
item_groups_eval[item_train_freq_t >= HEAD_THRESH] = 2                     # HEAD

# Phân tầng User (0=inactive/cold, 1=active, 2=super_active)
user_activity_npy = np.load(download_hf("gold/gold_user_activity_group.npy", local_dir=CFG["CACHE_DIR"]))
user_groups_eval = torch.from_numpy(user_activity_npy).long().to(device)

print("[INFO] Đã chuẩn bị cờ phân tầng xong.")


--- [BƯỚC 5] ID MAPPING: CHUẨN BỊ TẬP TEST ---
[INFO] Đã load best model (epoch 12).


bronze/bronze_test.parquet/part-00000-e0(…):   0%|          | 0.00/31.1M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00001-e0(…):   0%|          | 0.00/31.0M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00002-e0(…):   0%|          | 0.00/45.1M [00:00<?, ?B/s]

bronze/bronze_test.parquet/part-00003-e0(…):   0%|          | 0.00/31.0M [00:00<?, ?B/s]

...

[INFO] Đang tải gold/gold_user_id_map.parquet...


gold/gold_user_id_map.parquet:   0%|          | 0.00/29.7M [00:00<?, ?B/s]

[INFO] Đang tải gold/gold_item_id_map.parquet...


gold/gold_item_id_map.parquet:   0%|          | 0.00/96.9M [00:00<?, ?B/s]

[INFO] Test interactions hợp lệ: 1,483,920

--- [BƯỚC 1] TRAINING ITEM MASKING (LỌC LỊCH SỬ) ---
[INFO] Đã tạo Masking Dictionary cho 1,483,920 users.

--- [BƯỚC 3] SEGMENTATION FLAGGING (CHUẨN BỊ NHÃN PHÂN TẦNG) ---
[INFO] Đang tải gold/gold_user_activity_group.npy...
[INFO] Đã chuẩn bị cờ phân tầng xong.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 15: PROTOCOL EVALUATION (MASKING, RE-RANKING & SEGMENT METRICS)
# ═══════════════════════════════════════════════════════════════════════════════
def evaluate_protocol(user_final_all, item_final_all, test_users, test_items,
                      item_train_freq_t, num_items, train_user_dict,
                      user_propensity_t, lambda_val, item_groups, user_groups,
                      Ks=[20, 40], device="cuda"):
    model.eval()

    # [BƯỚC 4] Chuẩn bị Popularity Penalty cho Re-ranking
    # Thay vì dùng log1p thô, ta ép nó về khoảng [0, 1] bằng cách chia cho giá trị lớn nhất
    raw_penalty = torch.log1p(item_train_freq_t.float())
    max_penalty = torch.clamp(raw_penalty.max(), min=1e-8)
    base_item_penalty = raw_penalty / max_penalty

    # Khởi tạo dict lưu kết quả phân tầng
    # Item Segments: overall, cold, tail, head
    # User Segments: super_active, active, inactive
    categories = ["overall", "item_cold", "item_tail", "item_head",
                  "user_super", "user_active", "user_inactive"]
    results = {c: {**{f"hit@{k}": 0.0 for k in Ks}, **{f"ndcg@{k}": 0.0 for k in Ks}, "n": 0} for c in categories}

    recommended_mask = torch.zeros(num_items, dtype=torch.bool, device=device)
    max_K = max(Ks)
    ndcg_w = torch.tensor([1.0 / np.log2(k + 2) for k in range(max_K)], device=device, dtype=torch.float32)
    batch_sz = 256
    item_chunk_size = 65536 # [BƯỚC 2] Dùng Chunking cho All-item Ranking (tối ưu O(N))

    with torch.no_grad():
        for start in tqdm(range(0, len(test_users), batch_sz), desc=f"Eval (λ={lambda_val})", leave=False):
            end = min(start + batch_sz, len(test_users))
            batch_u = test_users[start:end]
            batch_p = test_items[start:end]
            curr_dtype = user_final_all.dtype

            # Tính hệ số cá nhân hóa cho penalty
            u_prop_f = (lambda_val * (1.0 - user_propensity_t[batch_u].to(curr_dtype)).unsqueeze(1)) if lambda_val > 0 else None

            # Lấy Masking Dictionary của batch
            batch_mask_rows, batch_mask_cols = [], []
            for local_i, u in enumerate(batch_u.cpu().tolist()):
                if u in train_user_dict:
                    items_u = train_user_dict[u].cpu().tolist()
                    batch_mask_rows.extend([local_i] * len(items_u))
                    batch_mask_cols.extend(items_u)
            bmr = np.array(batch_mask_rows, dtype=np.int32)
            bmc = np.array(batch_mask_cols, dtype=np.int32)

            chunk_vals, chunk_idxs = [], []

            # [BƯỚC 2] Thực hiện không gian xếp hạng All-Item bằng Chunking
            for cs in range(0, num_items, item_chunk_size):
                ce = min(cs + item_chunk_size, num_items)
                item_chunk = item_final_all[cs:ce]
                scores = torch.mm(user_final_all[batch_u], item_chunk.t())

                # [BƯỚC 1] Thực thi Training Item Masking (-1e4)
                if len(bmc) > 0:
                    valid = (bmc >= cs) & (bmc < ce)
                    if valid.any():
                        scores[bmr[valid], bmc[valid] - cs] = -1e4

                # [BƯỚC 4] Áp dụng Re-ranking Penalty
                if u_prop_f is not None:
                    scores.addmm_(u_prop_f, base_item_penalty[cs:ce].to(curr_dtype).unsqueeze(0), alpha=-1.0)

                c_v, c_i = torch.topk(scores, min(max_K, scores.shape[1]), dim=1)
                chunk_vals.append(c_v)
                chunk_idxs.append(c_i + cs)
                del scores

            # Trích xuất Top-K cuối cùng
            cat_vals = torch.cat(chunk_vals, dim=1)
            cat_idxs = torch.cat(chunk_idxs, dim=1)
            _, order = torch.topk(cat_vals, max_K, dim=1)
            top_idx = torch.gather(cat_idxs, 1, order)

            hits = (top_idx == batch_p.unsqueeze(1))
            ndcgs = hits.float() * ndcg_w.unsqueeze(0)
            recommended_mask[top_idx.flatten()] = True

            # [BƯỚC 3] Lấy cờ phân tầng cho batch
            batch_item_cats = item_groups[batch_p]
            batch_user_cats = user_groups[batch_u]

            # Ghi nhận kết quả theo ITEM Segments
            for cat_i, cname in enumerate(["item_cold", "item_tail", "item_head"]):
                m = (batch_item_cats == cat_i) if cname != "item_head" else (batch_item_cats == 2)
                if cname == "item_cold": m = (batch_item_cats == 0)
                if cname == "item_tail": m = (batch_item_cats == 1)
                if not m.any(): continue
                for k in Ks:
                    results[cname][f"hit@{k}"] += hits[m, :k].sum().item()
                    results[cname][f"ndcg@{k}"] += ndcgs[m, :k].sum().item()
                results[cname]["n"] += m.sum().item()

            # Ghi nhận kết quả theo USER Segments
            for cat_i, cname in enumerate(["user_inactive", "user_active", "user_super"]):
                m = (batch_user_cats == cat_i)
                if not m.any(): continue
                for k in Ks:
                    results[cname][f"hit@{k}"] += hits[m, :k].sum().item()
                    results[cname][f"ndcg@{k}"] += ndcgs[m, :k].sum().item()
                results[cname]["n"] += m.sum().item()

            # Ghi nhận Overall
            for k in Ks:
                results["overall"][f"hit@{k}"] += hits[:, :k].sum().item()
                results["overall"][f"ndcg@{k}"] += ndcgs[:, :k].sum().item()
            results["overall"]["n"] += len(batch_u)

    # Tổng hợp Metrics cuối cùng
    metrics = {}
    for cat in categories:
        n = max(results[cat]["n"], 1)
        for k in Ks:
            metrics[f"Recall@{k}_{cat}"] = results[cat][f"hit@{k}"] / n
            metrics[f"NDCG@{k}_{cat}"] = results[cat][f"ndcg@{k}"] / n

    tail_mask = (item_groups == 1)
    metrics["Coverage@20_tail"] = (tail_mask & recommended_mask).sum().item() / max(tail_mask.sum().item(), 1)
    return metrics

print("[INFO] Hàm evaluate_protocol (chuẩn 5 bước) đã sẵn sàng.")

[INFO] Hàm evaluate_protocol (chuẩn 5 bước) đã sẵn sàng.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 16: TÍNH TOÁN FINAL REPRESENTATIONS TRƯỚC KHI EVALUATE
# ═══════════════════════════════════════════════════════════════════════════════
print("\n--- TẠO FINAL REPRESENTATIONS ---")
model.eval()
gc.collect(); torch.cuda.empty_cache()

with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda")):
    # Refresh z_llm_all_cached với best model
    z_llm_all_final = precompute_z_llm(model, user_emb_llm, item_emb_llm, device)

    # Full GCN với gated fusion
    z_G_all = model.forward_gcn_gated(sparse_adj, z_llm_all_final, freq_all,
                                       niche_ratio_all=niche_ratio_all)
    user_G_all, item_G_all = z_G_all.split([num_users, num_items])

    # User representations
    user_L_chunks = []
    for s in range(0, num_users, 65536):
        e = min(s + 65536, num_users)
        user_L_chunks.append(model.project_text(user_emb_llm[s:e].float()))
    user_L_all = torch.cat(user_L_chunks, dim=0)

    user_final_all = F.normalize(
        model.get_final_repr(user_G_all, user_L_all), p=2, dim=1
    )

    # Item representations
    item_L_chunks = []
    for s in range(0, num_items, 65536):
        e = min(s + 65536, num_items)
        item_L_chunks.append(model.project_text(item_emb_llm[s:e].float()))
    item_L_all = torch.cat(item_L_chunks, dim=0)

    item_final_all = F.normalize(
        model.get_final_repr(item_G_all, item_L_all), p=2, dim=1
    )

del user_L_chunks, item_L_chunks, user_L_all, item_L_all
del z_G_all, user_G_all, item_G_all, z_llm_all_final
gc.collect(); torch.cuda.empty_cache()

print(f"[READY] user_final_all: {user_final_all.shape} | item_final_all: {item_final_all.shape}")


--- TẠO FINAL REPRESENTATIONS ---
[READY] user_final_all: torch.Size([1483920, 128]) | item_final_all: torch.Size([1610012, 128])


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 17: CHUẨN BỊ METADATA VÀ LỌC MẪU ĐÁNH GIÁ (SAMPLED EVALUATION)
# ═══════════════════════════════════════════════════════════════════════════════
import os, json
import torch
import numpy as np
import random
from tqdm.auto import tqdm

print("\n--- [1] CHUẨN BỊ METADATA ITEM GROUPS ---")
item_groups_npy = np.load(download_hf("gold/gold_item_popularity_group.npy"))
item_groups_t   = torch.from_numpy(item_groups_npy).long().to(device)

item_groups_eval = torch.zeros_like(item_groups_t)
item_groups_eval[item_groups_t == 2] = 1   # TAIL → 1
item_groups_eval[item_groups_t == 0] = 2   # HEAD → 2
del item_groups_npy; gc.collect()

print(f"\n--- [2] CHUẨN BỊ 100 MẪU CHO MỖI NHÃN USER ---")
test_user_labels = user_groups_eval[test_users_filt]
sampled_indices = []
g_eval = torch.Generator()
g_eval.manual_seed(42)

for label_idx, label_name in [(0, "INACTIVE"), (1, "ACTIVE"), (2, "SUPER_ACTIVE")]:
    valid_idx = torch.where(test_user_labels == label_idx)[0]
    n_samples = min(100, len(valid_idx))
    if n_samples > 0:
        perm = torch.randperm(len(valid_idx), generator=g_eval)[:n_samples]
        chosen_idx = valid_idx[perm]
        sampled_indices.append(chosen_idx)
        print(f"  Nhóm {label_name:<15}: Lấy {n_samples} interactions.")

final_test_idx = torch.cat(sampled_indices)
test_u_final = test_users_filt[final_test_idx].tolist()
test_i_final = test_items_filt[final_test_idx].tolist()

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 18: ĐÁNH GIÁ CHÍNH THỨC — MÔ HÌNH SAMPLED (1 POS + 100 NEG)
# ═══════════════════════════════════════════════════════════════════════════════
print("\n--- [3] TIỀN XỬ LÝ 100 NEGATIVES (CHỐNG RÒ RỈ LỊCH SỬ) ---")
num_items_total = item_final_all.shape[0]
random.seed(42)

eval_records = []
for u_idx, pos_item_idx in tqdm(zip(test_u_final, test_i_final), total=len(test_u_final), desc="Tạo Negative Samples"):
    u_history = train_user_dict.get(u_idx, [])
    neg_items = set()
    while len(neg_items) < 100:
        neg_item = random.randint(0, num_items_total - 1)
        if neg_item != pos_item_idx and neg_item not in u_history:
            neg_items.add(neg_item)

    pop_group_id = item_groups_eval[pos_item_idx].item()
    if pop_group_id == 0: pop_group = 'item_cold'
    elif pop_group_id == 1: pop_group = 'item_tail'
    else: pop_group = 'item_head'

    usr_group_id = user_groups_eval[u_idx].item()
    if usr_group_id == 0: usr_group = 'user_inactive'
    elif usr_group_id == 1: usr_group = 'user_active'
    else: usr_group = 'user_super'

    eval_records.append({
        'user_idx': u_idx, 'pos_item_idx': pos_item_idx,
        'neg_item_idxs': list(neg_items), 'pop_group': pop_group, 'usr_group': usr_group
    })

print("\n--- [4] TÍNH TOÁN BASE PENALTY (RE-RANKING) ---")
# Phục hồi mảng user_propensity_t từ Cell cũ của bạn
user_propensity_t = user_niche_t.clone()

# Giữ nguyên công thức tính Penalty của tác giả
raw_penalty = torch.log1p(item_train_freq_t.float())
max_penalty = torch.clamp(raw_penalty.max(), min=1e-8)
base_item_penalty = raw_penalty / max_penalty

def compute_sampled_metrics(records, lmd, k=20):
    if not records: return 0.0, 0.0
    hits, ndcg_sum = 0, 0.0
    for r in records:
        targets = [r['pos_item_idx']] + r['neg_item_idxs']
        u_emb = user_final_all[r['user_idx']].unsqueeze(0).to(device)
        i_embs = item_final_all[targets].to(device)

        # 1. Tính Dot Product cơ bản
        scores = torch.matmul(u_emb, i_embs.T).squeeze(0)

        # 2. Áp dụng Re-ranking Penalty nếu có
        if lmd > 0.0:
            u_prop = 1.0 - user_propensity_t[r['user_idx']].item()
            i_penalties = base_item_penalty[targets].to(device)
            # Trừ bớt điểm của các Item Phổ Biến (Head) để kích Tail lên
            scores = scores - (lmd * u_prop * i_penalties)

        _, top_indices = torch.topk(scores, k)

        if 0 in top_indices.tolist():
            hits += 1
            rank = top_indices.tolist().index(0)
            ndcg_sum += 1.0 / np.log2(rank + 2)

    return (hits / len(records)), (ndcg_sum / len(records))

print("\n--- [5] TUNING LAMBDA (TÌM THÔNG SỐ TỐT NHẤT CHO TAIL) ---")
lambda_candidates = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
best_lmd = 0.0
best_score = -1.0

print(f"{'Lambda':<8} | {'Overall R@20':<15} | {'Overall NDCG@20':<15}")
print("-" * 45)
for lmd in lambda_candidates:
    r, n = compute_sampled_metrics(eval_records, lmd)
    score = r + n
    print(f"{lmd:<8.3f} | {r:<15.5f} | {n:<15.5f}")
    if score > best_score:
        best_score = score
        best_lmd = lmd
print("-" * 45)
print(f"✅ Đã chọn Lambda tối ưu: {best_lmd:.3f}")

print("\n" + "═"*65)
print(f" BÁO CÁO KẾT QUẢ TA-RECMIND V2 (1 POS + 100 NEG | λ={best_lmd})")
print("═"*65)
metrics_dict = {}
metrics_dict['overall_r'], metrics_dict['overall_n'] = compute_sampled_metrics(eval_records, best_lmd)

for group_name in ['item_head', 'item_tail', 'item_cold']:
    recs = [r for r in eval_records if r['pop_group'] == group_name]
    metrics_dict[f'{group_name}_r'], metrics_dict[f'{group_name}_n'] = compute_sampled_metrics(recs, best_lmd)

for group_name in ['user_super', 'user_active', 'user_inactive']:
    recs = [r for r in eval_records if r['usr_group'] == group_name]
    metrics_dict[f'{group_name}_r'], metrics_dict[f'{group_name}_n'] = compute_sampled_metrics(recs, best_lmd)

print(f"  {'Nhóm':<22} | {'Recall@20':>10} | {'NDCG@20':>10}")
print(f"  {'-'*48}")
print(f"  {'Overall':<22} | {metrics_dict['overall_r']:>10.5f} | {metrics_dict['overall_n']:>10.5f}")
print(f"  --- ITEM GROUPS ---")
print(f"  {'Item Head':<22} | {metrics_dict['item_head_r']:>10.5f} | {metrics_dict['item_head_n']:>10.5f}")
print(f"  {'Item Tail':<22} | {metrics_dict['item_tail_r']:>10.5f} | {metrics_dict['item_tail_n']:>10.5f}")
print(f"  {'Item Cold':<22} | {metrics_dict['item_cold_r']:>10.5f} | {metrics_dict['item_cold_n']:>10.5f}")
print(f"  --- USER GROUPS ---")
print(f"  {'User Super Active':<22} | {metrics_dict['user_super_r']:>10.5f} | {metrics_dict['user_super_n']:>10.5f}")
print(f"  {'User Active':<22} | {metrics_dict['user_active_r']:>10.5f} | {metrics_dict['user_active_n']:>10.5f}")
print(f"  {'User Inactive':<22} | {metrics_dict['user_inactive_r']:>10.5f} | {metrics_dict['user_inactive_n']:>10.5f}")
print("═"*65)



--- [1] CHUẨN BỊ METADATA ITEM GROUPS ---
[INFO] Đang tải gold/gold_item_popularity_group.npy...

--- [2] CHUẨN BỊ 100 MẪU CHO MỖI NHÃN USER ---
  Nhóm INACTIVE       : Lấy 100 interactions.
  Nhóm ACTIVE         : Lấy 100 interactions.
  Nhóm SUPER_ACTIVE   : Lấy 100 interactions.

--- [3] TIỀN XỬ LÝ 100 NEGATIVES (CHỐNG RÒ RỈ LỊCH SỬ) ---


Tạo Negative Samples:   0%|          | 0/300 [00:00<?, ?it/s]


--- [4] TÍNH TOÁN BASE PENALTY (RE-RANKING) ---

--- [5] TUNING LAMBDA (TÌM THÔNG SỐ TỐT NHẤT CHO TAIL) ---
Lambda   | Overall R@20    | Overall NDCG@20
---------------------------------------------
0.000    | 0.34333         | 0.15344        
0.050    | 0.30333         | 0.13535        
0.100    | 0.27000         | 0.12065        
0.150    | 0.24667         | 0.10644        
0.200    | 0.24333         | 0.09966        
0.300    | 0.18333         | 0.08040        
0.500    | 0.13333         | 0.05296        
---------------------------------------------
✅ Đã chọn Lambda tối ưu: 0.000

═════════════════════════════════════════════════════════════════
 BÁO CÁO KẾT QUẢ TA-RECMIND V2 (1 POS + 100 NEG | λ=0.0)
═════════════════════════════════════════════════════════════════
  Nhóm                   |  Recall@20 |    NDCG@20
  ------------------------------------------------
  Overall                |    0.34333 |    0.15344
  --- ITEM GROUPS ---
  Item Head              |    0.34914 |    

Đánh giá thêm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ĐÁNH GIÁ 3000 MẪU + LƯU TRỮ DEMO (RECALL@20 & RECALL@40)
# ═══════════════════════════════════════════════════════════════════════════════
import os, json
import torch
import numpy as np
import random
from tqdm.auto import tqdm

print("\n--- [1] CHUẨN BỊ METADATA ITEM GROUPS ---")
item_groups_npy = np.load(download_hf("gold/gold_item_popularity_group.npy"))
item_groups_t   = torch.from_numpy(item_groups_npy).long().to(device)

item_groups_eval = torch.zeros_like(item_groups_t)
item_groups_eval[item_groups_t == 2] = 1   # TAIL → 1
item_groups_eval[item_groups_t == 0] = 2   # HEAD → 2
del item_groups_npy; gc.collect()

print(f"\n--- [2] CHUẨN BỊ 1000 MẪU CHO MỖI NHÃN USER ---")
test_user_labels = user_groups_eval[test_users_filt]
sampled_indices = []
g_eval = torch.Generator()
g_eval.manual_seed(42)

for label_idx, label_name in [(0, "INACTIVE"), (1, "ACTIVE"), (2, "SUPER_ACTIVE")]:
    valid_idx = torch.where(test_user_labels == label_idx)[0]
    # NÂNG CẤP: Lấy 1000 mẫu thay vì 100
    n_samples = min(1000, len(valid_idx))
    if n_samples > 0:
        perm = torch.randperm(len(valid_idx), generator=g_eval)[:n_samples]
        chosen_idx = valid_idx[perm]
        sampled_indices.append(chosen_idx)
        print(f"  Nhóm {label_name:<15}: Lấy {n_samples} interactions.")

final_test_idx = torch.cat(sampled_indices)
test_u_final = test_users_filt[final_test_idx].tolist()
test_i_final = test_items_filt[final_test_idx].tolist()

print("\n--- [3] TIỀN XỬ LÝ 100 NEGATIVES CHO 3000 MẪU ---")
num_items_total = item_final_all.shape[0]
random.seed(42)

eval_records = []
for u_idx, pos_item_idx in tqdm(zip(test_u_final, test_i_final), total=len(test_u_final), desc="Tạo Negative Samples"):
    u_history = train_user_dict.get(u_idx, [])
    neg_items = set()
    while len(neg_items) < 100:
        neg_item = random.randint(0, num_items_total - 1)
        if neg_item != pos_item_idx and neg_item not in u_history:
            neg_items.add(neg_item)

    pop_group_id = item_groups_eval[pos_item_idx].item()
    if pop_group_id == 0: pop_group = 'item_cold'
    elif pop_group_id == 1: pop_group = 'item_tail'
    else: pop_group = 'item_head'

    usr_group_id = user_groups_eval[u_idx].item()
    if usr_group_id == 0: usr_group = 'user_inactive'
    elif usr_group_id == 1: usr_group = 'user_active'
    else: usr_group = 'user_super'

    eval_records.append({
        'user_idx': u_idx, 'pos_item_idx': pos_item_idx,
        'neg_item_idxs': list(neg_items), 'pop_group': pop_group, 'usr_group': usr_group
    })

print("\n--- [4] ĐÁNH GIÁ FINAL INFERENCE (LAMBDA=0.0) ---")
def compute_sampled_metrics_multi_K(records, Ks=[20, 40]):
    if not records:
        return {k: {'r': 0.0, 'n': 0.0} for k in Ks}

    results = {k: {'hits': 0, 'ndcg_sum': 0.0} for k in Ks}
    max_k = max(Ks)

    for r in records:
        targets = [r['pos_item_idx']] + r['neg_item_idxs']
        u_emb = user_final_all[r['user_idx']].unsqueeze(0).to(device)
        i_embs = item_final_all[targets].to(device)

        scores = torch.matmul(u_emb, i_embs.T).squeeze(0)
        _, top_indices = torch.topk(scores, max_k)
        top_list = top_indices.tolist()

        for k in Ks:
            top_k_list = top_list[:k]
            if 0 in top_k_list:
                results[k]['hits'] += 1
                rank = top_k_list.index(0)
                results[k]['ndcg_sum'] += 1.0 / np.log2(rank + 2)

    n_recs = len(records)
    return {k: {'r': results[k]['hits']/n_recs, 'n': results[k]['ndcg_sum']/n_recs} for k in Ks}

# Tính toán các nhánh
metrics_dict = {}
metrics_dict['overall'] = compute_sampled_metrics_multi_K(eval_records)

for group in ['item_head', 'item_tail', 'item_cold', 'user_super', 'user_active', 'user_inactive']:
    # Item groups
    recs = [r for r in eval_records if r.get('pop_group') == group or r.get('usr_group') == group]
    metrics_dict[group] = compute_sampled_metrics_multi_K(recs)

print("\n" + "═"*85)
print(" BÁO CÁO KẾT QUẢ TA-RECMIND V2 (1 POS + 100 NEG | 3000 MẪU)")
print("═"*85)
print(f"  {'Nhóm':<20} | {'Recall@20':>12} | {'NDCG@20':>12} || {'Recall@40':>12} | {'NDCG@40':>12}")
print(f"  {'-'*80}")

def print_row(label, key):
    r20 = metrics_dict[key][20]['r']
    n20 = metrics_dict[key][20]['n']
    r40 = metrics_dict[key][40]['r']
    n40 = metrics_dict[key][40]['n']
    print(f"  {label:<20} | {r20:>12.5f} | {n20:>12.5f} || {r40:>12.5f} | {n40:>12.5f}")

print_row("Overall", "overall")
print(f"  --- ITEM GROUPS ---")
print_row("Item Head", "item_head")
print_row("Item Tail", "item_tail")
print_row("Item Cold", "item_cold")
print(f"  --- USER GROUPS ---")
print_row("User Super Active", "user_super")
print_row("User Active", "user_active")
print_row("User Inactive", "user_inactive")
print("═"*85)

print("\n--- [5] LƯU TRỮ MÔ HÌNH PHỤC VỤ DEMO ---")
# Tạo thư mục nếu chưa có
save_dir = CFG.get("SAVE_DIR", "/content/drive/MyDrive/tarecmind/weights")
os.makedirs(save_dir, exist_ok=True)
demo_path = os.path.join(save_dir, "tarecmind_demo_embeddings.pth")

# Lưu trực tiếp Final Representations
# Giao diện Demo chỉ cần load 2 mảng này và làm phép nhân (Dot Product)
torch.save({
    'user_embeddings': user_final_all.cpu(),
    'item_embeddings': item_final_all.cpu(),
}, demo_path)

print(f"✅ Đã xuất bản bộ nhúng phục vụ giao diện Demo tại: {demo_path}")
print("💡 Gợi ý cho App Demo: Khi người dùng truy cập, App chỉ cần lấy vector của họ trong 'user_embeddings', nhân (dot product) với toàn bộ 'item_embeddings' và lấy Top 20 điểm cao nhất là xong. Tốc độ sẽ chưa tới 1 mili-giây!")



--- [1] CHUẨN BỊ METADATA ITEM GROUPS ---
[INFO] Đang tải gold/gold_item_popularity_group.npy...

--- [2] CHUẨN BỊ 1000 MẪU CHO MỖI NHÃN USER ---
  Nhóm INACTIVE       : Lấy 1000 interactions.
  Nhóm ACTIVE         : Lấy 1000 interactions.
  Nhóm SUPER_ACTIVE   : Lấy 1000 interactions.

--- [3] TIỀN XỬ LÝ 100 NEGATIVES CHO 3000 MẪU ---


Tạo Negative Samples:   0%|          | 0/3000 [00:00<?, ?it/s]


--- [4] ĐÁNH GIÁ FINAL INFERENCE (LAMBDA=0.0) ---

═════════════════════════════════════════════════════════════════════════════════════
 BÁO CÁO KẾT QUẢ TA-RECMIND V2 (1 POS + 100 NEG | 3000 MẪU)
═════════════════════════════════════════════════════════════════════════════════════
  Nhóm                 |    Recall@20 |      NDCG@20 ||    Recall@40 |      NDCG@40
  --------------------------------------------------------------------------------
  Overall              |      0.34400 |      0.15962 ||      0.53067 |      0.19755
  --- ITEM GROUPS ---
  Item Head            |      0.34696 |      0.16232 ||      0.53373 |      0.20029
  Item Tail            |      0.33910 |      0.14729 ||      0.53633 |      0.18728
  Item Cold            |      0.32743 |      0.15124 ||      0.50442 |      0.18715
  --- USER GROUPS ---
  User Super Active    |      0.35000 |      0.16680 ||      0.53700 |      0.20496
  User Active          |      0.35000 |      0.16273 ||      0.53600 |      0.20042
 

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 19: TỰ NGẮT KẾT NỐI (tiết kiệm Compute Units)
# ═══════════════════════════════════════════════════════════════════════════════
import time

print("Toàn bộ pipeline đã hoàn tất và lưu lên Drive.")
print("Hệ thống sẽ ngắt kết nối sau 10 giây...")
time.sleep(10)

try:
    from google.colab import runtime
    runtime.unassign()
except:
    print("[INFO] Không ở môi trường Colab — bỏ qua ngắt kết nối.")

Toàn bộ pipeline đã hoàn tất và lưu lên Drive.
Hệ thống sẽ ngắt kết nối sau 10 giây...


In [ ]:
# ── Build random eval sample ───────────────────────────────────────────────
eval_groups = {}

val_u = val_edges_t[0].cpu()
val_i = val_edges_t[1].cpu()

# nếu muốn sample nhỏ (ví dụ 1000 user)
#N = min(CFG["EVAL_STRAT_N"], len(val_u))
N = min(100, len(val_u))
rng = np.random.default_rng(2024)
idx = rng.choice(len(val_u), size=N, replace=False)

eval_groups["RANDOM"] = (
    val_u[idx],
    val_i[idx]
)

print(f"[INFO] Random eval sample: {N} users")


[INFO] Random eval sample: 100 users


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL: ĐÁNH GIÁ RANKING — Recall@K & NDCG@K tại K=20 và K=40
#       Thiết lập: 100 mẫu tiêu cực (negatives) mỗi người dùng
#
# Luồng đánh giá:
#   1. Với mỗi (user, pos_item) trong tập val:
#      - Lấy ngẫu nhiên 100 items tiêu cực (chưa tương tác trong train)
#      - Pool đánh giá = {pos_item} ∪ {100 negatives} = 101 mục
#      - Tính điểm cosine cho toàn bộ pool
#      - Xác định rank của pos_item trong pool → tính Recall@K, NDCG@K
#   2. Tổng hợp theo nhóm phân tầng (stratified groups) và overall
#   3. In kết quả đầy đủ tại K=20 và K=40
# ═══════════════════════════════════════════════════════════════════════════════

NUM_NEG_EVAL   = 100          # số mẫu tiêu cực cho mỗi người dùng
EVAL_K_LIST    = [20, 40]     # các mốc K cần đánh giá
EVAL_SEED      = 2024         # seed cố định để tái lập kết quả


# ── Bước 1: Xây dựng tập đã tương tác trong train cho từng user ──────────────
# Dùng để lọc false negative khi lấy mẫu tiêu cực
print("[INFO] Xây dựng user_train_set để lọc false negative...")
_t0 = time.time()

# train_edge_index[0] = user ids, train_edge_index[1] = item ids (offset đã trừ)
_train_u = train_edge_index[0].cpu().numpy()
_train_i = train_edge_index[1].cpu().numpy()

from collections import defaultdict
user_train_items: dict = defaultdict(set)
for u, i in zip(_train_u, _train_i):
    user_train_items[int(u)].add(int(i))

print(f"[INFO] Đã lập chỉ mục train set trong {time.time()-_t0:.1f}s")
del _train_u, _train_i


# ── Bước 2: Hàm lấy mẫu tiêu cực cho evaluation ─────────────────────────────
def sample_eval_negatives(
    user_idx: int,
    pos_item: int,
    num_items: int,
    user_train_items: dict,
    n_neg: int = NUM_NEG_EVAL,
    rng: np.random.Generator = None,
) -> np.ndarray:
    """
    Lấy n_neg items tiêu cực cho user_idx, đảm bảo:
      - pos_item không nằm trong danh sách âm
      - Các items trong tập train của user bị loại trừ (tránh false negative)
    Trả về mảng numpy gồm n_neg item ids.
    """
    if rng is None:
        rng = np.random.default_rng(EVAL_SEED)

    exclude = user_train_items.get(user_idx, set()) | {pos_item}
    negs = []
    # Lấy mẫu có thay thế với lượng dư để đảm bảo đủ sau khi lọc
    candidates = rng.integers(0, num_items, size=n_neg * 4)
    for c in candidates:
        if int(c) not in exclude:
            negs.append(int(c))
            if len(negs) == n_neg:
                break

    # Trường hợp hiếm: nếu không đủ, bổ sung bằng vòng lặp
    while len(negs) < n_neg:
        c = rng.integers(0, num_items)
        if int(c) not in exclude:
            negs.append(int(c))

    return np.array(negs, dtype=np.int64)


# ── Bước 3: Hàm đánh giá chính — 100 negatives, K = {20, 40} ─────────────────
def evaluate_with_negatives(
    model,
    adj,
    z_llm_all_cached,
    freq_all,
    user_emb_llm,
    item_emb_llm,
    eval_groups: dict,
    item_train_freq_t,
    user_train_items: dict,
    num_users: int,
    num_items: int,
    k_list: list  = EVAL_K_LIST,
    n_neg: int    = NUM_NEG_EVAL,
    device: str   = "cuda",
    seed: int     = EVAL_SEED,
) -> dict:
    """
    Đánh giá xếp hạng với pool = {1 pos + n_neg negatives} mỗi user.

    Trả về dict lồng nhau:
      results[group_name][K] = {"Recall": float, "NDCG": float, "n": int}
    """
    rng = np.random.default_rng(seed)
    model.eval()

    # ── Precompute toàn bộ item representations ───────────────────────────────
    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda")):
        z_G_all = model.forward_gcn_gated(adj, z_llm_all_cached, freq_all)
        _, item_G_all = z_G_all.split([num_users, num_items])

        # Project LLM của items theo chunk để tránh OOM
        item_L_parts = []
        for s in range(0, num_items, 32768):
            e = min(s + 32768, num_items)
            item_L_parts.append(
                model.project_text(item_emb_llm[s:e].to(device))
            )
        item_L_all = torch.cat(item_L_parts, dim=0)

        # Biểu diễn cuối cùng của items — L2 normalize
        item_final = F.normalize(
            model.get_final_repr(item_G_all, item_L_all), p=2, dim=1
        ).cpu()   # Chuyển về CPU để lập chỉ mục nhanh theo pool

    # ── Khởi tạo bộ đếm ──────────────────────────────────────────────────────
    # results_raw[group][K] = {"hit": float, "ndcg": float, "n": int}
    results_raw = {}
    for gname in list(eval_groups.keys()) + ["OVERALL"]:
        results_raw[gname] = {K: {"hit": 0.0, "ndcg": 0.0, "n": 0}
                               for K in k_list}

    overall_raw = {K: {"hit": 0.0, "ndcg": 0.0, "n": 0} for K in k_list}

    # ── Vòng lặp đánh giá theo từng nhóm ─────────────────────────────────────
    for gname, (g_users, g_items) in eval_groups.items():
        if g_users is None:
            continue

        g_u = g_users.cpu().numpy()
        g_i = g_items.cpu().numpy()

        for idx in range(len(g_u)):
            u_idx    = int(g_u[idx])
            pos_item = int(g_i[idx])

            # Lấy mẫu 100 negatives
            neg_items = sample_eval_negatives(
                u_idx, pos_item, num_items, user_train_items, n_neg, rng
            )

            # Pool = pos_item + negatives; pos_item ở vị trí 0
            pool_ids = np.concatenate([[pos_item], neg_items])   # [n_neg+1]
            pool_t   = torch.from_numpy(pool_ids).long()

            # Tính biểu diễn user
            with torch.no_grad(), torch.amp.autocast(
                "cuda", enabled=(device == "cuda")
            ):
                u_llm_v = user_emb_llm[u_idx:u_idx + 1].to(device)
                u_G_v   = z_G_all[u_idx:u_idx + 1]
                u_L_v   = model.project_text(u_llm_v)
                u_final = F.normalize(
                    model.get_final_repr(u_G_v, u_L_v), p=2, dim=1
                ).cpu()   # [1, d]

            # Tính điểm cosine cho toàn bộ pool
            pool_emb = item_final[pool_t]        # [n_neg+1, d]
            scores   = (u_final * pool_emb).sum(dim=1)   # [n_neg+1]

            # pos_item ở vị trí 0; rank = số items có score > pos_score + 1
            pos_score = scores[0]
            rank = int((scores > pos_score).sum().item()) + 1

            # Cập nhật chỉ số tại mỗi mốc K
            for K in k_list:
                hit  = 1 if rank <= K else 0
                ndcg = (1.0 / np.log2(rank + 1)) if rank <= K else 0.0
                results_raw[gname][K]["hit"]  += hit
                results_raw[gname][K]["ndcg"] += ndcg
                results_raw[gname][K]["n"]    += 1
                overall_raw[K]["hit"]         += hit
                overall_raw[K]["ndcg"]        += ndcg
                overall_raw[K]["n"]           += 1

    # ── Tổng hợp kết quả cuối ─────────────────────────────────────────────────
    final_results = {}

    for gname in eval_groups:
        final_results[gname] = {}
        for K in k_list:
            n = results_raw[gname][K]["n"]
            final_results[gname][K] = {
                "Recall": results_raw[gname][K]["hit"]  / max(n, 1),
                "NDCG":   results_raw[gname][K]["ndcg"] / max(n, 1),
                "n":      n,
            }

    final_results["OVERALL"] = {}
    for K in k_list:
        n = overall_raw[K]["n"]
        final_results["OVERALL"][K] = {
            "Recall": overall_raw[K]["hit"]  / max(n, 1),
            "NDCG":   overall_raw[K]["ndcg"] / max(n, 1),
            "n":      n,
        }

    return final_results


# ── Bước 4: Hàm in kết quả đẹp ────────────────────────────────────────────────
def print_eval_results(epoch: int, results: dict, k_list: list = EVAL_K_LIST):
    """
    In bảng kết quả đánh giá tại tất cả mốc K.
    results: dict trả về từ evaluate_with_negatives()
    """
    sep = "─" * 72
    print(f"\n{'═'*72}")
    print(f"  [EVAL] Epoch {epoch:03d} | Pool = 1 pos + {NUM_NEG_EVAL} neg mỗi user")
    print(f"{'═'*72}")

    for K in k_list:
        print(f"\n  ── K = {K} {'─'*55}")
        header = f"  {'Nhóm':<18} | {'Recall@'+str(K):>12} | {'NDCG@'+str(K):>12} | {'N':>6}"
        print(header)
        print(f"  {sep}")
        for gname, kdict in results.items():
            if K not in kdict:
                continue
            v = kdict[K]
            marker = " ★" if gname == "OVERALL" else ""
            print(
                f"  {gname:<18} | {v['Recall']:>12.4f} | {v['NDCG']:>12.4f}"
                f" | {v['n']:>6}{marker}"
            )

    print(f"\n{'═'*72}\n")


# ── Bước 5: Chạy đánh giá ─────────────────────────────────────────────────────
# Gọi hàm đánh giá đầy đủ và in kết quả
# Sử dụng best model checkpoint để đánh giá cuối cùng

print("[INFO] Bắt đầu đánh giá ranking (100 negatives, K=20 & K=40)...")
_eval_t0 = time.time()

eval_results_final = evaluate_with_negatives(
    model           = model,
    adj             = sparse_adj,
    z_llm_all_cached= z_llm_all_cached,
    freq_all        = freq_all,
    user_emb_llm    = user_emb_llm,
    item_emb_llm    = item_emb_llm,
    eval_groups     = eval_groups,
    item_train_freq_t = item_train_freq_t,
    user_train_items  = user_train_items,
    num_users       = num_users,
    num_items       = num_items,
    k_list          = EVAL_K_LIST,
    n_neg           = NUM_NEG_EVAL,
    device          = device,
    seed            = EVAL_SEED,
)

_eval_elapsed = time.time() - _eval_t0
print(f"[INFO] Đánh giá hoàn tất trong {_eval_elapsed:.1f}s")

# In bảng kết quả
print_eval_results(epoch=resume_epoch, results=eval_results_final)

# ── Bước 6: Lưu kết quả vào history ──────────────────────────────────────────
_eval_record = {"epoch": resume_epoch, "eval_neg": NUM_NEG_EVAL}
for K in EVAL_K_LIST:
    overall = eval_results_final.get("OVERALL", {}).get(K, {})
    _eval_record[f"Recall@{K}_overall"] = overall.get("Recall", 0.0)
    _eval_record[f"NDCG@{K}_overall"]   = overall.get("NDCG",   0.0)

print("[INFO] Kết quả đánh giá (OVERALL):")
for K in EVAL_K_LIST:
    print(
        f"  Recall@{K} = {_eval_record[f'Recall@{K}_overall']:.4f} | "
        f"NDCG@{K}   = {_eval_record[f'NDCG@{K}_overall']:.4f}"
    )


[INFO] Xây dựng user_train_set để lọc false negative...
[INFO] Đã lập chỉ mục train set trong 11.4s
[INFO] Bắt đầu đánh giá ranking (100 negatives, K=20 & K=40)...
[INFO] Đánh giá hoàn tất trong 0.7s

════════════════════════════════════════════════════════════════════════
  [EVAL] Epoch 042 | Pool = 1 pos + 100 neg mỗi user
════════════════════════════════════════════════════════════════════════

  ── K = 20 ───────────────────────────────────────────────────────
  Nhóm               |    Recall@20 |      NDCG@20 |      N
  ────────────────────────────────────────────────────────────────────────
  RANDOM             |       0.4600 |       0.2362 |    100
  OVERALL            |       0.4600 |       0.2362 |    100 ★

  ── K = 40 ───────────────────────────────────────────────────────
  Nhóm               |    Recall@40 |      NDCG@40 |      N
  ────────────────────────────────────────────────────────────────────────
  RANDOM             |       0.5900 |       0.2636 |    100
  OVERALL 